# OpenTelemetry spans inside experiments 🔭

This notebook shows how to run a Datadog LLM Observability experiment whose task emits OpenTelemetry GenAI spans through the Python tracer.

The important setup is `DD_TRACE_OTEL_ENABLED=1` plus registering `ddtrace.opentelemetry.TracerProvider` as the OpenTelemetry provider. With that provider active, spans created with `opentelemetry.trace` inside an experiment task are parented under the experiment span.

References:

- [OpenTelemetry instrumentation for LLM Observability](https://docs.datadoghq.com/llm_observability/instrumentation/otel_instrumentation/?tab=python#experiments)
- [Using OpenTelemetry spans inside experiments](https://docs.datadoghq.com/llm_observability/experiments/setup/#using-opentelemetry-spans-inside-experiments)


## 0. Set up

Load credentials, enable the OpenTelemetry bridge, and initialize LLM Observability.

> **Notebook tip:** `DD_TRACE_OTEL_ENABLED` must be set before `ddtrace` is imported. This cell also calls `set_tracer_provider(DatadogTracerProvider())` because notebooks usually do not run through `ddtrace-run` preload. If you already imported `ddtrace` in this kernel without the flag, restart the kernel and run this cell first.


In [ ]:
import os
import json
from typing import Any, Dict

from dotenv import load_dotenv

# Load environment variables from the .env file.
load_dotenv(override=True)

# Required for ddtrace to act as the OpenTelemetry TracerProvider.
# Set this before importing ddtrace or ddtrace.llmobs.
os.environ.setdefault("DD_TRACE_OTEL_ENABLED", "1")

from opentelemetry import trace
from opentelemetry.trace import SpanKind, Status, StatusCode, set_tracer_provider

from ddtrace.opentelemetry import TracerProvider as DatadogTracerProvider
from ddtrace.llmobs import EvaluatorResult, LLMObs
from openai import OpenAI

# In scripts launched with ddtrace-run or ddtrace.auto, preload.py registers this
# provider when DD_TRACE_OTEL_ENABLED=1. Notebooks usually run without preload,
# so register it explicitly. Re-running this cell in the same kernel may log an
# OpenTelemetry warning that the provider is already set; restart the kernel if
# you need to change providers.
set_tracer_provider(DatadogTracerProvider())

datadog_app_key = os.getenv("DD_APPLICATION_KEY") or os.getenv("DD_APP_KEY")
missing = [
    name
    for name, value in {
        "DD_API_KEY": os.getenv("DD_API_KEY"),
        "DD_APPLICATION_KEY or DD_APP_KEY": datadog_app_key,
        "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    }.items()
    if not value
]
if missing:
    raise RuntimeError(f"Missing required environment variables: {', '.join(missing)}")

LLMObs.enable(
    api_key=os.getenv("DD_API_KEY"),
    app_key=datadog_app_key,
    site=os.getenv("DD_SITE", "datadoghq.com"),
    project_name="otel-experiments-demo",
    ml_app="otel-experiments-demo",
    agentless_enabled=True,
)

oai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
otel_tracer = trace.get_tracer("llmobs.experiments.otel")

print("DD_TRACE_OTEL_ENABLED=", os.getenv("DD_TRACE_OTEL_ENABLED"))
print("OpenTelemetry tracer provider:", type(trace.get_tracer_provider()).__module__)


## 1. Create a small dataset

The experiment task below asks OpenAI to answer capital-city questions. The expected output is intentionally short so the evaluators can stay deterministic.


In [ ]:
dataset = LLMObs.create_dataset(
    dataset_name="otel-capitals-demo",
    description="A small dataset for testing OpenTelemetry spans inside LLMObs experiments.",
    records=[
        {
            "input_data": {"question": "Which country has Paris as its capital?"},
            "expected_output": "France",
            "metadata": {"topic": "capitals"},
        },
        {
            "input_data": {"question": "Which country has Tokyo as its capital?"},
            "expected_output": "Japan",
            "metadata": {"topic": "capitals"},
        },
        {
            "input_data": {"question": "Which country has Ottawa as its capital?"},
            "expected_output": "Canada",
            "metadata": {"topic": "capitals"},
        },
    ],
)

dataset.as_dataframe()


## 2. Define a task that creates an OpenTelemetry span

`answer_capital_with_otel_span` is a normal experiment task: it receives `input_data` and `config`, calls a model, and returns the output.

The only OpenTelemetry-specific part is the `with otel_tracer.start_as_current_span(...)` block. The span uses OpenTelemetry GenAI semantic attribute names such as `gen_ai.system`, `gen_ai.request.model`, `gen_ai.input.messages`, and `gen_ai.output.messages`. Because the task runs inside an experiment and `DD_TRACE_OTEL_ENABLED=1` is set, this OTel span appears as a child of the experiment span.


In [ ]:
def _as_otel_chat_messages(messages: list[dict[str, str]]) -> str:
    """Serialize OpenAI-style messages in the GenAI semantic convention shape."""
    return json.dumps(
        [
            {
                "role": message["role"],
                "parts": [{"type": "text", "content": message["content"]}],
            }
            for message in messages
        ]
    )


def answer_capital_with_otel_span(input_data: Dict[str, Any], config: Dict[str, Any]) -> str:
    question = input_data["question"]
    model = config.get("model", "gpt-4o-mini")
    temperature = config.get("temperature", 0)
    max_tokens = config.get("max_tokens", 20)
    messages = [
        {
            "role": "system",
            "content": config.get(
                "system_prompt",
                "Answer with only the country name. Do not include punctuation.",
            ),
        },
        {"role": "user", "content": question},
    ]

    with otel_tracer.start_as_current_span(
        "openai.chat.completions",
        kind=SpanKind.CLIENT,
    ) as span:
        span.set_attribute("gen_ai.operation.name", "chat")
        span.set_attribute("gen_ai.system", "openai")
        span.set_attribute("gen_ai.request.model", model)
        span.set_attribute("gen_ai.request.temperature", float(temperature))
        span.set_attribute("gen_ai.request.max_tokens", int(max_tokens))
        span.set_attribute("gen_ai.input.messages", _as_otel_chat_messages(messages))

        try:
            response = oai_client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                max_tokens=max_tokens,
            )
            answer = (response.choices[0].message.content or "").strip()

            if getattr(response, "model", None):
                span.set_attribute("gen_ai.response.model", response.model)

            usage = getattr(response, "usage", None)
            if usage is not None:
                token_attributes = {
                    "gen_ai.usage.input_tokens": getattr(usage, "prompt_tokens", None),
                    "gen_ai.usage.output_tokens": getattr(usage, "completion_tokens", None),
                    "gen_ai.usage.total_tokens": getattr(usage, "total_tokens", None),
                }
                for attribute, value in token_attributes.items():
                    if value is not None:
                        span.set_attribute(attribute, int(value))

            span.set_attribute(
                "gen_ai.output.messages",
                _as_otel_chat_messages([{"role": "assistant", "content": answer}]),
            )
            span.set_status(Status(StatusCode.OK))
            return answer
        except Exception as exc:
            span.record_exception(exc)
            span.set_status(Status(StatusCode.ERROR, str(exc)))
            raise


## 3. Add evaluators

The evaluators receive the input, output, and expected output for each dataset record. They do not need to know whether the task was instrumented with the Datadog SDK or OpenTelemetry.


In [ ]:
def contains_expected_answer(input_data, output_data, expected_output):
    expected = str(expected_output).strip().lower()
    actual = str(output_data).strip().lower()
    passed = expected in actual
    return EvaluatorResult(
        value=passed,
        reasoning=(
            f"Expected answer '{expected_output}' was found in '{output_data}'."
            if passed
            else f"Expected answer '{expected_output}' was not found in '{output_data}'."
        ),
        assessment="pass" if passed else "fail",
        tags={"metric": "contains_expected_answer"},
    )


def answer_is_concise(input_data, output_data, expected_output):
    word_count = len(str(output_data).split())
    passed = word_count <= 12
    return EvaluatorResult(
        value=passed,
        reasoning=f"Answer used {word_count} words; expected at most 12.",
        assessment="pass" if passed else "fail",
        tags={"metric": "answer_is_concise"},
    )


## 4. Run the experiment

Running the experiment creates a Datadog experiment span for each dataset row. Each row also includes the OpenTelemetry `openai.chat.completions` child span created inside the task.

For local notebooks, `agentless_enabled=True` sends LLM Observability data directly to Datadog. If you want experiment span ↔ APM trace correlation, run through a Datadog Agent and use the default `agentless_enabled=False` instead.


In [ ]:
experiment = LLMObs.experiment(
    name="otel-openai-capitals",
    dataset=dataset,
    task=answer_capital_with_otel_span,
    evaluators=[contains_expected_answer, answer_is_concise],
    config={
        "model": "gpt-4o-mini",
        "temperature": 0,
        "max_tokens": 20,
        "system_prompt": "Answer with only the country name. Do not include punctuation.",
        "instrumentation": "opentelemetry",
        "otel_span_name": "openai.chat.completions",
    },
    description="Experiment task emits OpenTelemetry GenAI spans through the Python tracer.",
)

results = experiment.run(jobs=3)
experiment.url


## 5. Inspect results

Open the experiment URL above, then drill into a record trace. You should see:

1. The experiment/task span generated by LLM Observability.
2. A child span named `openai.chat.completions`.
3. GenAI attributes like `gen_ai.system=openai`, `gen_ai.request.model`, token usage, input messages, and output messages on that child span.

If the OpenTelemetry span is missing, confirm that the kernel was restarted after setting `DD_TRACE_OTEL_ENABLED=1` and that the setup cell ran before importing `ddtrace`.


In [ ]:
try:
    results.as_dataframe()
except AttributeError:
    results
